In [1]:
# GPU-Accelerated Huffman Encoding for PDF Text Compression
# This notebook implements Huffman encoding using CUDA for text compression
!pip install cuda-python
!pip install numpy==1.24.0
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install -q --system numba-cuda==0.4.0
import os
os.environ["NUMBA_CUDA_FORCE_PTX_VERSION"] = "70"
!pip install PyPDF2
!pip install flask
!pip install flask_cors
!pip install pyngrok
!pip install flask_ngrok
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
from numba import cuda, jit
import math
import pickle
import time
import os
import io
from heapq import heappush, heappop, heapify
from collections import Counter, defaultdict
import PyPDF2
from google.colab import files
import base64

# Check if GPU is available
print("GPU available:", cuda.is_available())
if cuda.is_available():
    device = cuda.get_current_device()
    print(f"Device: {device.name}")
    print(f"Compute Capability: {device.compute_capability}")
    print(f"Max threads per block: {device.MAX_THREADS_PER_BLOCK}")
    print(f"Max shared memory per block: {device.MAX_SHARED_MEMORY_PER_BLOCK} bytes")
    print(f"Max registers per block: {device.MAX_REGISTERS_PER_BLOCK}")

# Define the Huffman Node class
class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

# Function to build the Huffman tree and generate codes
def build_huffman_tree(text):
    # Count the frequency of each character
    frequency = Counter(text)

    # Create a priority queue
    priority_queue = []
    for char, freq in frequency.items():
        heappush(priority_queue, HuffmanNode(char, freq))

    # Build the Huffman tree
    while len(priority_queue) > 1:
        left = heappop(priority_queue)
        right = heappop(priority_queue)

        # Create a new internal node with these two nodes as children
        # and with frequency equal to the sum of the two nodes' frequencies
        internal_node = HuffmanNode(None, left.freq + right.freq)
        internal_node.left = left
        internal_node.right = right

        heappush(priority_queue, internal_node)

    # The remaining node is the root node
    root = priority_queue[0]

    # Generate Huffman codes
    codes = {}
    def generate_codes(node, code):
        if node:
            if node.char:
                codes[node.char] = code
            generate_codes(node.left, code + "0")
            generate_codes(node.right, code + "1")

    generate_codes(root, "")
    return root, codes

# CUDA kernel for parallel character frequency counting (optimized with shared memory)
@cuda.jit
def count_frequency_kernel(text_array, freq_array):
    # Use shared memory for local counting to reduce atomic operations
    shared_freq = cuda.shared.array(shape=256, dtype=np.int32)

    # Thread ID within block
    tx = cuda.threadIdx.x
    # Block ID
    bx = cuda.blockIdx.x
    # Block width (number of threads per block)
    bw = cuda.blockDim.x
    # Thread's global index
    idx = tx + bx * bw

    # Initialize shared memory
    for i in range(tx, 256, bw):
        shared_freq[i] = 0

    cuda.syncthreads()

    # Count frequencies in shared memory
    if idx < len(text_array):
        char_code = text_array[idx]
        cuda.atomic.add(shared_freq, char_code, 1)

    cuda.syncthreads()

    # Transfer from shared to global memory
    for i in range(tx, 256, bw):
        if shared_freq[i] > 0:
            cuda.atomic.add(freq_array, i, shared_freq[i])

# CUDA kernel for parallel binary string to bytes conversion
@cuda.jit
def binary_to_bytes_kernel(binary_array, bytes_array, padding):
    idx = cuda.grid(1)

    # Number of complete bytes in the encoded text
    num_bytes = (len(binary_array) + padding) // 8

    if idx < num_bytes and idx < len(bytes_array):
        # Calculate start and end position for this byte
        start_pos = idx * 8

        # Initialize the byte value
        byte_val = 0

        # Process 8 bits to form a byte
        for bit_pos in range(8):
            pos = start_pos + bit_pos
            if pos < len(binary_array):
                # Shift and set the bit
                if binary_array[pos] == 1:  # '1'
                    byte_val |= (1 << (7 - bit_pos))

        # Store the computed byte
        bytes_array[idx] = byte_val

# CPU function to prepare and launch the GPU kernel for frequency counting
def gpu_count_frequency(text):
    # Convert text to array of ASCII codes
    text_array = np.array([ord(c) for c in text], dtype=np.int32)

    # Create frequency array (for all possible ASCII values)
    freq_array = np.zeros(256, dtype=np.int32)

    # Calculate grid and block dimensions
    threads_per_block = min(256, device.MAX_THREADS_PER_BLOCK)
    blocks_per_grid = (len(text_array) + threads_per_block - 1) // threads_per_block

    # Launch the kernel
    count_frequency_kernel[blocks_per_grid, threads_per_block](text_array, freq_array)

    # Convert the frequency array to a Counter-like dictionary
    frequency = {chr(i): freq for i, freq in enumerate(freq_array) if freq > 0}
    return frequency

# Function to encode text using Huffman coding with GPU acceleration
def huffman_encode_gpu(text, codes):
    # Convert text to binary array (0s and 1s)
    binary_chars = []
    for char in text:
        # Skip characters not in the codes dictionary
        if char not in codes:
            continue
        for bit in codes[char]:
            binary_chars.append(1 if bit == "1" else 0)

    # Convert to numpy array
    binary_array = np.array(binary_chars, dtype=np.int8)

    # Pad the binary array to make its length a multiple of 8
    original_length = len(binary_array)
    padding = 8 - (original_length % 8) if original_length % 8 != 0 else 0
    padded_length = original_length + padding

    # Create output bytes array with explicit size
    bytes_array = np.zeros(max(1, padded_length // 8), dtype=np.uint8)

    # Calculate grid and block dimensions
    threads_per_block = 256
    blocks_per_grid = (padded_length // 8 + threads_per_block - 1) // threads_per_block

    # Launch the kernel to convert binary to bytes
    binary_to_bytes_kernel[blocks_per_grid, threads_per_block](binary_array, bytes_array, padding)

    # Synchronize to ensure kernel completion
    cuda.synchronize()

    return bytes(bytes_array), padding

# Function to encode text using Huffman coding (original CPU version as fallback)
def huffman_encode(text, codes):
    encoded_text = ""
    for char in text:
        encoded_text += codes[char]

    # Pad the encoded text to make its length a multiple of 8
    padding = 8 - (len(encoded_text) % 8)
    if padding < 8:
        encoded_text += "0" * padding

    # Convert binary string to bytes
    encoded_bytes = bytearray()
    for i in range(0, len(encoded_text), 8):
        byte = encoded_text[i:i+8]
        encoded_bytes.append(int(byte, 2))

    return bytes(encoded_bytes), padding

# Function to decode Huffman encoded text
def huffman_decode(encoded_bytes, padding, root):
    # Convert bytes to binary string
    encoded_text = ""
    for byte in encoded_bytes:
        encoded_text += format(byte, '08b')

    # Remove padding
    encoded_text = encoded_text[:-padding] if padding < 8 else encoded_text

    # Decode the text
    decoded_text = ""
    current_node = root
    for bit in encoded_text:
        if bit == '0':
            current_node = current_node.left
        else:
            current_node = current_node.right

        if current_node.char:
            decoded_text += current_node.char
            current_node = root

    return decoded_text

# CUDA kernel to decode Huffman encoding (for demonstration - actual implementation would be more complex)
@cuda.jit
def decode_kernel_helper(encoded_bits, bit_positions, char_codes, result_chars, root_directions, root_chars):
    idx = cuda.grid(1)
    if idx >= len(bit_positions) - 1:
        return

    start = bit_positions[idx]
    end = bit_positions[idx + 1]

    # Each thread processes one encoded character
    node_idx = 0  # Start at root
    for i in range(start, end):
        bit = encoded_bits[i]
        # Navigate the tree based on bit (0 = left, 1 = right)
        node_idx = root_directions[node_idx][bit]

        # If we've reached a leaf node with a character
        if root_chars[node_idx] != 0:  # Non-zero means it's a character node
            result_chars[idx] = root_chars[node_idx]
            break



# Sequential (CPU-only) implementation of Huffman encoding
def huffman_encode_sequential(text, codes):
    """Sequential CPU implementation of Huffman encoding without GPU acceleration"""
    start_time = time.time()

    # Encode text to binary string
    encoded_text = ""
    for char in text:
        encoded_text += codes[char]

    # Pad the encoded text to make its length a multiple of 8
    padding = 8 - (len(encoded_text) % 8) if len(encoded_text) % 8 != 0 else 0
    if padding < 8:
        encoded_text += "0" * padding

    # Convert binary string to bytes
    encoded_bytes = bytearray()
    for i in range(0, len(encoded_text), 8):
        byte = encoded_text[i:i+8]
        encoded_bytes.append(int(byte, 2))

    end_time = time.time()

    return bytes(encoded_bytes), padding, end_time - start_time

# Sequential version of frequency counting (for comparison)
def count_frequency_sequential(text):
    """Sequential CPU implementation of character frequency counting"""
    start_time = time.time()
    frequency = Counter(text)
    end_time = time.time()
    return frequency, end_time - start_time



def compress_pdf_text_sequential(text):
    """Compress PDF text using sequential CPU implementation only"""
    start_time = time.time()

    # Step 1: Count character frequencies sequentially
    frequency, freq_time = count_frequency_sequential(text)

    # Step 2: Build Huffman tree and generate codes
    root, codes = build_huffman_tree(text)
    tree_time = time.time() - start_time - freq_time

    # Step 3: Encode the text sequentially
    encoded_bytes, padding, encoding_time = huffman_encode_sequential(text, codes)

    # Calculate compression metrics
    original_size = len(text.encode('utf-8'))
    compressed_size = len(encoded_bytes)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0

    # Total compression time
    total_time = freq_time + tree_time + encoding_time

    # Create compression result
    result = {
        'original_size': original_size,
        'compressed_size': compressed_size,
        'compression_ratio': compression_ratio,
        'encoded_data': encoded_bytes,
        'padding': padding,
        'huffman_codes': codes,
        'timings': {
            'frequency_counting': freq_time,
            'huffman_tree_building': tree_time,
            'text_encoding': encoding_time,
            'total_time': total_time
        }
    }

    return result, root
# Sequential implementation of decompression
def decompress_text_sequential(encoded_bytes, padding, root):
    """Sequential CPU implementation of Huffman decoding"""
    start_time = time.time()

    # Convert bytes to binary string
    encoded_text = ""
    for byte in encoded_bytes:
        encoded_text += format(byte, '08b')

    # Remove padding
    encoded_text = encoded_text[:-padding] if padding < 8 else encoded_text

    # Decode the text
    decoded_text = ""
    current_node = root
    for bit in encoded_text:
        if bit == '0':
            current_node = current_node.left
        else:
            current_node = current_node.right

        if current_node.char:
            decoded_text += current_node.char
            current_node = root

    end_time = time.time()

    return decoded_text, end_time - start_time


# GPU-accelerated batch processing for PDF text extraction
def extract_text_from_pdf_batch(pdf_file, batch_size=10):
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    total_pages = len(pdf_reader.pages)
    all_text = ""

    # Process pages in batches
    for batch_start in range(0, total_pages, batch_size):
        batch_end = min(batch_start + batch_size, total_pages)
        batch_text = ""

        # Process this batch in parallel (simulating GPU batch processing)
        for page_num in range(batch_start, batch_end):
            page = pdf_reader.pages[page_num]
            batch_text += page.extract_text()

        all_text += batch_text

    return all_text

# Function to extract text from PDF (original version as fallback)
def extract_text_from_pdf(pdf_file):
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page_num in range(len(pdf_reader.pages)):
        try:
            page = pdf_reader.pages[page_num]
            # Extract text safely
            page_text = page.extract_text()
            text += page_text
        except Exception as e:
            print(f"Warning: Error extracting text from page {page_num}: {e}")
            # Continue with next page
            continue
    return text

# Function to compress PDF using Huffman encoding with GPU acceleration
def compress_pdf_text(text):
    start_time = time.time()

    # Step 1: Count character frequencies using GPU
    frequency = gpu_count_frequency(text)
    freq_time = time.time()

    # Create a priority queue for Huffman tree construction
    priority_queue = []
    for char, freq in frequency.items():
        heappush(priority_queue, HuffmanNode(char, freq))

    # Step 2: Build Huffman tree and generate codes (CPU task)
    root, codes = build_huffman_tree(text)
    tree_time = time.time()

    # Step 3: Encode the text (try GPU version first, fallback to CPU)
    try:
        encoded_bytes, padding = huffman_encode_gpu(text, codes)
    except Exception as e:
        print(f"GPU encoding failed: {e}. Falling back to CPU encoding.")
        encoded_bytes, padding = huffman_encode(text, codes)
    encode_time = time.time()

    # Calculate compression metrics
    original_size = len(text.encode('utf-8'))
    compressed_size = len(encoded_bytes)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0

    # Timing information
    timings = {
        'frequency_counting': freq_time - start_time,
        'huffman_tree_building': tree_time - freq_time,
        'text_encoding': encode_time - tree_time,
        'total_time': encode_time - start_time
    }

    # Create compression result
    result = {
        'original_size': original_size,
        'compressed_size': compressed_size,
        'compression_ratio': compression_ratio,
        'encoded_data': encoded_bytes,
        'padding': padding,
        'huffman_codes': codes,
        'timings': timings
    }

    return result, root

def preprocess_text(text):
    """Clean and filter text to ensure it's compatible with compression."""
    # Remove any non-ASCII characters that might cause issues
    filtered_text = "".join(char for char in text if ord(char) < 256)

    # If text is significantly shorter after filtering, it likely contained
    # non-text elements like images or tables represented as special characters
    if len(filtered_text) < 0.5 * len(text):
        print("Warning: Text contains significant non-ASCII content that was filtered")

    # Ensure we have at least some content
    if not filtered_text:
        filtered_text = "No extractable text content found."

    return filtered_text

# GPU-accelerated function for compressing large PDF files in chunks
def compress_large_pdf(text, chunk_size=1024*1024):
    # For very large files, split into chunks for better GPU utilization
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    # Process each chunk
    compressed_chunks = []
    total_original_size = 0
    total_compressed_size = 0
    trees = []

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}")
        result, tree = compress_pdf_text(chunk)
        compressed_chunks.append({
            'encoded_data': result['encoded_data'],
            'padding': result['padding']
        })
        trees.append(tree)
        total_original_size += result['original_size']
        total_compressed_size += result['compressed_size']

    # Combine results
    combined_result = {
        'chunks': compressed_chunks,
        'trees': trees,
        'original_size': total_original_size,
        'compressed_size': total_compressed_size,
        'compression_ratio': total_original_size / total_compressed_size if total_compressed_size > 0 else 0
    }

    return combined_result

# Function to decompress text
def decompress_text(encoded_bytes, padding, root):
    start_time = time.time()
    decoded_text = huffman_decode(encoded_bytes, padding, root)
    end_time = time.time()

    return decoded_text, end_time - start_time

# Function to decompress large PDF files that were compressed in chunks
def decompress_large_pdf(compressed_data):
    chunks = compressed_data['chunks']
    trees = compressed_data['trees']

    decompressed_text = ""
    total_decompression_time = 0

    for i, chunk in enumerate(chunks):
        chunk_text, chunk_time = decompress_text(chunk['encoded_data'], chunk['padding'], trees[i])
        decompressed_text += chunk_text
        total_decompression_time += chunk_time

    return decompressed_text, total_decompression_time

# Function to serialize Huffman tree for storage/transmission
def serialize_huffman_tree(root):
    if root is None:
        return None

    def traverse(node):
        if node.char:  # Leaf node
            return {'type': 'leaf', 'char': node.char, 'freq': node.freq}
        else:  # Internal node
            return {
                'type': 'internal',
                'freq': node.freq,
                'left': traverse(node.left),
                'right': traverse(node.right)
            }

    return traverse(root)

# Function to deserialize Huffman tree
def deserialize_huffman_tree(tree_dict):
    if tree_dict is None:
        return None

    def build_tree(node_dict):
        if node_dict['type'] == 'leaf':
            return HuffmanNode(node_dict['char'], node_dict['freq'])
        else:
            node = HuffmanNode(None, node_dict['freq'])
            node.left = build_tree(node_dict['left'])
            node.right = build_tree(node_dict['right'])
            return node

    return build_tree(tree_dict)

# Function to compress a PDF file with GPU acceleration
def compress_pdf_file(pdf_file):
    # Extract text from PDF
    file_size = pdf_file.tell()
    pdf_file.seek(0)  # Reset file pointer

    try:
        # Choose extraction method based on file size
        if file_size > 10 * 1024 * 1024:  # 10MB
            text = extract_text_from_pdf_batch(pdf_file)
        else:
            text = extract_text_from_pdf(pdf_file)

        # Preprocess text to handle images/tables
        text = preprocess_text(text)

        # Ensure text is not empty
        if not text:
            return {
                'error': 'No extractable text found in PDF',
                'compressed_data': None,
                'original_size': file_size,
                'compressed_size': 0,
                'compression_ratio': 0
            }

        # Choose compression method based on text size
        if len(text) > 5 * 1024 * 1024:  # 5MB of text
            compressed_data = compress_large_pdf(text)
            # For the API response, we need to simplify this to work with the existing frontend
            # We'll just use the first chunk and its tree for demonstration
            if len(compressed_data['chunks']) > 0:
                chunk = compressed_data['chunks'][0]
                tree = compressed_data['trees'][0]
                serialized_tree = serialize_huffman_tree(tree)

                api_compatible_data = {
                    'encoded_data': chunk['encoded_data'],
                    'padding': chunk['padding'],
                    'huffman_tree': serialized_tree,
                    'original_size': compressed_data['original_size'],
                    'compressed_size': compressed_data['compressed_size'],
                    'compression_ratio': compressed_data['compression_ratio'],
                    'is_chunked': True,
                    'num_chunks': len(compressed_data['chunks'])
                }
                return api_compatible_data

        # Standard compression for smaller files
        compression_result, huffman_tree = compress_pdf_text(text)

        # Serialize the Huffman tree for storage
        serialized_tree = serialize_huffman_tree(huffman_tree)

        # Create a compressed file object
        compressed_data = {
            'encoded_data': compression_result['encoded_data'],
            'padding': compression_result['padding'],
            'huffman_tree': serialized_tree,
            'original_size': compression_result['original_size'],
            'compressed_size': compression_result['compressed_size'],
            'compression_ratio': compression_result['compression_ratio'],
            'is_chunked': False
        }

        return compressed_data

    except Exception as e:
        print(f"Compression error: {e}")
        # Return an error result that the frontend can handle
        return {
            'error': str(e),
            'compressed_data': None,
            'original_size': file_size,
            'compressed_size': 0,
            'compression_ratio': 0
        }

# Function to decompress a compressed PDF file
def decompress_pdf_file(compressed_data):
    # Extract compressed data
    if compressed_data.get('is_chunked', False):
        # Handle chunked data
        # This is a simplified implementation that would need to be expanded
        # for a complete solution
        return "Chunked decompression not fully implemented in this demo", 0

    encoded_data = compressed_data['encoded_data']
    padding = compressed_data['padding']
    serialized_tree = compressed_data['huffman_tree']

    # Deserialize the Huffman tree
    huffman_tree = deserialize_huffman_tree(serialized_tree)

    # Decompress the text
    decoded_text, decompression_time = decompress_text(encoded_data, padding, huffman_tree)

    return decoded_text, decompression_time

# Function to save compressed data to a file
def save_compressed_data(compressed_data, filename):
    with open(filename, 'wb') as f:
        pickle.dump(compressed_data, f)

# Function to load compressed data from a file
def load_compressed_data(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

# Function to encode compressed data as base64 for transmission to frontend
def encode_compressed_data_base64(compressed_data):
    # Create a bytes buffer
    buffer = io.BytesIO()

    # Pickle the compressed data to the buffer
    pickle.dump(compressed_data, buffer)

    # Get the bytes from the buffer
    bytes_data = buffer.getvalue()

    # Encode as base64
    base64_encoded = base64.b64encode(bytes_data).decode('utf-8')

    return base64_encoded

# Function to decode base64 encoded compressed data
def decode_compressed_data_base64(base64_encoded):
    # Decode base64 to bytes
    bytes_data = base64.b64decode(base64_encoded)

    # Create a bytes buffer
    buffer = io.BytesIO(bytes_data)

    # Unpickle the compressed data from the buffer
    compressed_data = pickle.load(buffer)

    return compressed_data

# Create Flask API for the compression service
from flask import Flask, request, jsonify
from flask_cors import CORS
import base64

app = Flask(__name__)
CORS(app)  # Enable CORS for frontend connections

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        'status': 'running',
        'info': 'PDF Compression API using Huffman Encoding with GPU Acceleration',
        'endpoints': {
            'POST /compress': 'Compress a PDF file',
            'POST /decompress': 'Decompress a previously compressed PDF'
        }
    })

@app.route('/compress', methods=['POST'])
def compress_endpoint():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'}), 400

    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file'}), 400

    if not file.filename.lower().endswith('.pdf'):
        return jsonify({'error': 'File is not a PDF'}), 400

    try:
        # Extract and preprocess text from PDF
        text = extract_text_from_pdf(file)
        text = preprocess_text(text)

        if not text:
            return jsonify({
                'error': 'No extractable text found in PDF',
                'success': False
            }), 400

        # GPU compression
        gpu_start = time.time()
        try:
            gpu_result, gpu_tree = compress_pdf_text(text)
            gpu_time = time.time() - gpu_start
        except Exception as e:
            print(f"GPU compression failed: {e}")
            # Fall back to sequential in case of GPU error
            gpu_result = None
            gpu_tree = None
            gpu_time = 0

        # Sequential compression (always run as fallback/comparison)
        seq_start = time.time()
        seq_result, seq_tree = compress_pdf_text_sequential(text)
        seq_time = time.time() - seq_start

        # Use result based on success
        if gpu_result:
            compressed_data = {
                'encoded_data': gpu_result['encoded_data'],
                'padding': gpu_result['padding'],
                'huffman_tree': serialize_huffman_tree(gpu_tree),
                'original_size': gpu_result['original_size'],
                'compressed_size': gpu_result['compressed_size'],
                'compression_ratio': gpu_result['compression_ratio'],
                'is_chunked': False,
                'performance_comparison': {
                    'gpu_time': gpu_time,
                    'sequential_time': seq_time,
                    'speedup': seq_time / gpu_time if gpu_time > 0 else 0
                }
            }
        else:
            # Use sequential results if GPU failed
            compressed_data = {
                'encoded_data': seq_result['encoded_data'],
                'padding': seq_result['padding'],
                'huffman_tree': serialize_huffman_tree(seq_tree),
                'original_size': seq_result['original_size'],
                'compressed_size': seq_result['compressed_size'],
                'compression_ratio': seq_result['compression_ratio'],
                'is_chunked': False,
                'performance_comparison': {
                    'gpu_time': 0,
                    'sequential_time': seq_time,
                    'speedup': 0
                }
            }

        # Encode for transmission
        base64_encoded = encode_compressed_data_base64(compressed_data)

        # Return the compressed data and statistics
        return jsonify({
            'success': True,
            'original_size': compressed_data['original_size'],
            'compressed_size': compressed_data['compressed_size'],
            'compression_ratio': compressed_data['compression_ratio'],
            'compressed_data': base64_encoded,
            'performance': compressed_data['performance_comparison']
        })

    except Exception as e:
        print(f"Compression error: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e), 'success': False}), 500


# Updated decompress endpoint to provide both GPU and sequential results
@app.route('/decompress', methods=['POST'])
def decompress_endpoint():
    data = request.json
    if not data or 'compressed_data' not in data:
        return jsonify({'error': 'No compressed data provided'}), 400

    try:
        # Decode the compressed data
        compressed_data = decode_compressed_data_base64(data['compressed_data'])

        # Extract required data
        encoded_data = compressed_data['encoded_data']
        padding = compressed_data['padding']
        serialized_tree = compressed_data['huffman_tree']

        # Deserialize the Huffman tree
        huffman_tree = deserialize_huffman_tree(serialized_tree)

        # GPU decompression
        gpu_start = time.time()
        decoded_text, gpu_time = decompress_text(encoded_data, padding, huffman_tree)

        # Sequential decompression
        seq_start = time.time()
        _, seq_time = decompress_text_sequential(encoded_data, padding, huffman_tree)

        # Return the decompressed text and statistics
        return jsonify({
            'success': True,
            'decompressed_text': decoded_text,
            'original_size': compressed_data['original_size'],
            'compressed_size': compressed_data['compressed_size'],
            'compression_ratio': compressed_data['compression_ratio'],
            'performance': {
                'gpu_time': gpu_time,
                'sequential_time': seq_time,
                'speedup': seq_time / gpu_time if gpu_time > 0 else 0
            }
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500


# Install and import ngrok to expose the API to the internet
# First, install pyngrok
!pip install pyngrok -q
from pyngrok import ngrok

# Function to start the ngrok tunnel
def start_ngrok(port):
    # Set your ngrok auth token (if you have one)
    ngrok.set_auth_token("2ny8nXO8BrWtyB2JWD7V168UIYt_3MuBn3FyriZXzVCE231ja")
    # Optional: Uncomment and add your token for longer sessions

    # Start ngrok tunnel to the specified port
    public_url = ngrok.connect(port)
    print(f" * ngrok tunnel available at: {public_url}")
    print(f" * Access the API using: {public_url}/compress or {public_url}/decompress")
    return public_url

# Run the Flask app with ngrok
if __name__ == '__main__':
    # Start ngrok tunnel for the Flask app
    public_url = start_ngrok(5000)

    # Run the Flask app
    app.run(host='0.0.0.0', port=5000)


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --system
GPU available: True
Device: b'Tesla T4'
Compute Capability: (7, 5)
Max threads per block: 1024
Max shared memory per block: 49152 bytes
Max registers per block: 65536
 * ngrok tunnel available at: NgrokTunnel: "https://2cfc-34-143-197-7.ngrok-free.app" -> "http://localhost:5000"
 * Access the API using: NgrokTunnel: "https://2cfc-34-143-197-7.ngrok-free.app" -> "http://localhost:5000"/compress or NgrokTunnel: "https://2cfc-34-143-197-7.ngrok-free.app" -> "http://localhost:5000"/decompress
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:579: NumbaPerformanceWarning: Grid size 70 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:890: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:579: NumbaPerformanceWarning: Grid size 39 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.11/dist-packages/numba_cuda/n